# 1.a. Basic Input Analysis

Before building a system for hierarchical multi-label classification, it is essential to understand the dataset and its components. This notebook includes exploring the training and test corpus, the list of product classes, the class hierarchy, and the associated keywords.

| File | What it contains | Role in pipeline |
| :--- | :--- | :--- |
| `train/train_corpus.txt` | 29,487 unlabeled product review texts | Input corpus for generating silver labels + training |
| `test/test_corpus.txt` | 19,658 unlabeled review texts | Model inference for Kaggle submission |
| `classes.txt` | List of 531 product categories | Label space definition |
| `class_hierarchy.txt` | Parent $\rightarrow$ child relationships between classes (DAG) | Label propagation + GNN graph construction |
| `class_related_keywords.txt` | Keyword list per class | Used to generate initial silver labels |
| `dummy_baseline.ipynb` | Example Kaggle submission format | Helps ensure your output CSV format is correct |
| `submission.csv` | Example submission file | Reference structure for predictions |

In [1]:
# Relevant imports
from pathlib import Path
import numpy as np
from collections import defaultdict, Counter
import re

# Setup initial filepaths
ROOT = Path("project_release")

TRAIN_CORPUS_PATH = ROOT / "Amazon_products" / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "Amazon_products" / "test" / "test_corpus.txt"

CLASSES_PATH = ROOT / "Amazon_products" / "classes.txt"
HIERARCHY_PATH = ROOT / "Amazon_products" / "class_hierarchy.txt"
KEYWORDS_PATH = ROOT / "Amazon_products" / "class_related_keywords.txt"

In [2]:
# train_corpus.txt
# Represents the reviews that we will train the model on
# review id / review - structure

with open(TRAIN_CORPUS_PATH, "r", encoding="utf8") as f:
    reviews = []
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        reviews.append(line.strip().split("\t", 1)[1])

lengths = np.array([len(r.split()) for r in reviews])

print("Total reviews:", len(lengths))
print("Mean length:", np.mean(lengths))
print("Median length:", np.median(lengths))
print("Std dev:", np.std(lengths))
print("Min length:", np.min(lengths))
print("Max length:", np.max(lengths))

for p in [50, 75, 90, 95, 99]:
    print(f"{p}th percentile:", np.percentile(lengths, p))

# test_corpus.txt

with open(TEST_CORPUS_PATH, "r") as f:
    for _ in range(5):
        print(next(f).strip())


0	omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on the bloodpressure effects of my various exercises , food consumption , and relaxation or stress levels .
1	natural factors whey factors chocolate works well , but there is a lot of dead space in the container when you first open it up . the container comes 3 4 4 5 full and the rest in empty space .
2	clif bar builder 's bar , 2 . 4 ounce bars i love the peanut butter builder 's bars . while amazon is great for so many things , a trip to tj 's is too good to pass up . a little cup of coffee , a sample or two and into the cart with some fresh vegetables and whatever else is irresistible that day . life does n't get much better . if tj 's were n't local ( perish the thought ! ) , i 'd order the bars from amazon . with amazon 's sales volume , 2 day fedex delivery and a sheltered place for 

In [3]:
# classes.txt
# Represents the different categories that we will match each review with
# Represents all product categories. Each review can belong to multiple classes, and if it belongs to a sub-class, it must also belong to its parent class.
# Notice broad variation. Assuming amazon products which can vary drastically in function and theme

with open(CLASSES_PATH, "r") as f:
    classes = [line.strip() for line in f]

print("Total classes:", len(classes))
print("\nFirst 15 classes:")
for c in classes[:15]:
    print(c)

"""
Example -
grocery_gourmet_food
 ├─ meat_poultry
 │   └─ jerky
toys_games
 ├─ games
 │   ├─ puzzles
 │   │   └─ jigsaw_puzzles
 │   └─ board_games
beverages
 └─ juices
beauty
 └─ makeup
     └─ nails
arts_crafts
 └─ drawing_painting_supplies
"""


Total classes: 531

First 15 classes:
0	grocery_gourmet_food
1	meat_poultry
2	jerky
3	toys_games
4	games
5	puzzles
6	jigsaw_puzzles
7	board_games
8	beverages
9	juices
10	beauty
11	makeup
12	nails
13	arts_crafts
14	drawing_painting_supplies


'\nExample -\ngrocery_gourmet_food\n ├─ meat_poultry\n │   └─ jerky\ntoys_games\n ├─ games\n │   ├─ puzzles\n │   │   └─ jigsaw_puzzles\n │   └─ board_games\nbeverages\n └─ juices\nbeauty\n └─ makeup\n     └─ nails\narts_crafts\n └─ drawing_painting_supplies\n'

In [4]:
# class_hierarchy.txt
# parent_index / child_index - structure
# It is simply mapping the hierarchical structure internally

with open(HIERARCHY_PATH, "r") as f:
    for _ in range(15):
        print(next(f).strip())

graph = defaultdict(list)

with open(HIERARCHY_PATH, "r") as f:
    for line in f:
        parent, child = map(int, line.strip().split("\t"))
        graph[parent].append(child)

print("Children of class 0:", graph[0])

0	1
0	8
0	208
0	211
0	213
0	216
0	229
0	255
0	265
0	218
0	271
0	277
0	249
0	288
0	313
Children of class 0: [1, 8, 208, 211, 213, 216, 229, 255, 265, 218, 271, 277, 249, 288, 313, 357]


In [5]:
# class_related_keywords.txt
# So exactly as the name of the file suggests, this file contains relevant keywords for any class
# This essentially creates a bridge between the class and the review. 

with open(KEYWORDS_PATH, "r") as f:
    for _ in range(10):
        print(next(f).strip())


# Below to confirm keyword duplicates exists. This means that the system might have to rank them somehow
class_keywords = {}
all_keywords = []

with open(KEYWORDS_PATH, "r") as f:
    for line in f:
        class_name, keywords_str = line.strip().split(":")
        keywords = keywords_str.split(",")
        class_keywords[class_name] = keywords
        all_keywords.extend(keywords)

for cls, kws in class_keywords.items():
    dup = [k for k, c in Counter(kws).items() if c > 1]
    if dup:
        print(f"Duplicate keywords in class {cls}: {dup}")

keyword_to_classes = defaultdict(list)
for cls, kws in class_keywords.items():
    for kw in kws:
        keyword_to_classes[kw].append(cls)

multi_class_keywords = {kw: cls_list for kw, cls_list in keyword_to_classes.items() if len(cls_list) > 1}
print("Keywords appearing in multiple classes:", multi_class_keywords)


grocery_gourmet_food:snacks,condiments,beverages,specialty_foods,spices,cooking_oils,baking_ingredients,gourmet_chocolates,artisanal_cheeses,organic_foods
meat_poultry:butcher,cuts,marination,grilling,roasting,seasoning,halal,organic,deli,marbling
jerky:beef,turkey,chicken,venison,buffalo,kangaroo,elk,ostrich,bison,spicy
toys_games:board_games,puzzles,action_figures,building_blocks,dolls,outdoor_toys,educational_toys,card_games,remote_control_toys,plush_toys
games:board_games,card_games,tabletop_games,party_games,roleplaying_games,video_games,strategy_games,family_games,word_games,dice_games
puzzles:jigsaw_puzzles,brain_teasers,puzzle_accessories,puzzle_storage,puzzle_mats,puzzle_glue,puzzle_organizers,puzzle_books,puzzle_magazines,puzzle_competitions
jigsaw_puzzles:interlocking_pieces,puzzle_boards,puzzle_glue,puzzle_storage,puzzle_frames,puzzle_rolls,puzzle_organizers,puzzle_tables,puzzle_sleeves,puzzle_sorting_trays
board_games:board_game_accessories,strategy_games,cooperative_games

# 1.b. Data Preparation

Before we can generate silver labels and train models, the raw dataset needs to be converted into structures that are easily readable and indexable by the system. This involves:

Mapping classes to indices and creating bidirectional dictionaries for quick lookups.

Converting the hierarchy into parent→child and child→parent mappings for hierarchical reasoning.

Organizing class-related keywords into dictionaries for efficient keyword matching.

Loading and preprocessing corpora into consistent, sanitized text formats for training and evaluation.

These steps prepare the dataset for downstream processing, ensuring that both the model and any automatic labeling methods can efficiently access and use the data.

In [6]:
# Step 1: Load and sanitize class names
# Idea here is to map name of classes to indices

classes = []
with open(CLASSES_PATH, "r", encoding="utf8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            idx, name = parts
            classes.append(name.strip().lower())

idx_to_class = {i: name for i, name in enumerate(classes)}
class_to_idx = {name: i for i, name in enumerate(classes)}

print("Total classes:", len(classes))
print("Class index 0:", idx_to_class[0])
print("Index of 'jerky':", class_to_idx.get('jerky', "Not found"))




Total classes: 531
Class index 0: grocery_gourmet_food
Index of 'jerky': 2


In [7]:
# Step 2: Load and Sanitize Hierarchy
# Convert raw parent-child index pairs into dictionaries for easy traversal:
# - children_map[parent_idx] gives all child indices
# - parents_map[child_idx] gives all parent indices

children_map = defaultdict(list)
parents_map = defaultdict(list)

with open(HIERARCHY_PATH, "r", encoding="utf8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            parent_idx, child_idx = map(int, parts)
            children_map[parent_idx].append(child_idx)
            parents_map[child_idx].append(parent_idx)

print("Children of class 0:", [idx_to_class[i] for i in children_map[0]])
print("Parents of class 2:", [idx_to_class[i] for i in parents_map[2]])


Children of class 0: ['meat_poultry', 'beverages', 'gourmet_gifts', 'sauces_dips', 'breakfast_foods', 'pantry_staples', 'fresh_flowers_live_indoor_plants', 'breads_bakery', 'candy_chocolate', 'cooking_baking_supplies', 'snack_food', 'meat_seafood', 'herbs', 'baby_food', 'dairy_eggs', 'produce']
Parents of class 2: ['meat_poultry']


In [8]:
# Step 3: Load class-related keywords
# Sanitize keywords

class_keywords = {}

with open(KEYWORDS_PATH, "r", encoding="utf8") as f:
    for line in f:
        parts = line.strip().split(":")
        cls = parts[0].strip().lower()
        kws = parts[1] if len(parts) > 1 else ""
        class_keywords[cls] = [k.strip().lower() for k in kws.split(",") if k]

print("Keywords for 'jerky':", class_keywords.get('jerky', []))
print("Keywords for 'beverages':", class_keywords.get('beverages', []))

Keywords for 'jerky': ['beef', 'turkey', 'chicken', 'venison', 'buffalo', 'kangaroo', 'elk', 'ostrich', 'bison', 'spicy']
Keywords for 'beverages': ['coffee', 'tea', 'energy_drinks', 'soft_drinks', 'bottled_water', 'juices', 'sports_drinks', 'smoothies', 'iced_tea', 'coconut_water']


In [9]:
# Step 4: Load and Preprocess Corpora
# Clean and sanitize review texts for both training and test data. Lowercasing may slightly lose meaning (e.g., "Apple" vs "apple") but this is negligible overall.

def load_corpus_plaintext(path):
    with open(path, "r", encoding="utf8") as f:
        return {i: line.strip() for i, line in enumerate(f)}

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_pid2text = load_corpus_plaintext(TRAIN_CORPUS_PATH)
test_pid2text = load_corpus_plaintext(TEST_CORPUS_PATH)

train_pid2text = {pid: preprocess_text(text) for pid, text in train_pid2text.items()}
test_pid2text = {pid: preprocess_text(text) for pid, text in test_pid2text.items()}

def dict2list(id2text):
    ids = list(id2text.keys())
    texts = [id2text[i] for i in ids]
    return ids, texts

train_ids, train_texts = dict2list(train_pid2text)
test_ids, test_texts = dict2list(test_pid2text)

print("First train review:", train_texts[0])
print("First test review:", test_texts[0])


First train review: 0 omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use it is nice to have immediate feedback on the bloodpressure effects of my various exercises food consumption and relaxation or stress levels
First test review: 0 conair cs15tcs professional straight styles straightening iron woah sure this straightener looks like all the other crappy straightners in the world but there s a twist to this one it is my first straightner and i ve had it for about 7 months i bought it only because i was desperate for a cheap straightener because my hair is very thick long wavy i m looking for a new straighner right now but until then this one is doing just fine if it works for me it will work for you
